## Coleta Spark

In [ ]:
import sys
import logging
import requests
import boto3
from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext

# -- Logging --
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -- Glue init --
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME"])
    job_name = args["JOB_NAME"]
except:
    args = {}
    job_name = "local_test"
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(job_name, args)

# -- Configuracoes --
s3 = boto3.client("s3")

BUCKET       = "datalake-pnad-covid-prod"
BASE_URL_DADOS = "https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_PNAD_COVID19/Microdados/Dados/"
BASE_URL_DOC   = "https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_PNAD_COVID19/Microdados/Documentacao/"

MICRODADOS = [
    "PNAD_COVID_092020.zip",
    "PNAD_COVID_102020.zip",
    "PNAD_COVID_112020.zip",
]

DOCUMENTACAO = [
    "Dicionario_PNAD_COVID_092020_20220621.xls",
    "Dicionario_PNAD_COVID_102020_20220621.xls",
    "Dicionario_PNAD_COVID_112020_20220621.xls",
]

# -- Funcoes --
def baixar_e_subir_zip(nome_arquivo):
    """Baixa o ZIP de microdados do FTP e sobe intacto para data_input."""
    url = f"{BASE_URL_DADOS}{nome_arquivo}"
    logger.info(f"[MICRODADOS] Iniciando download: {url}")

    response = requests.get(url, timeout=300)
    response.raise_for_status()

    s3_key = f"data_input/microdados/{nome_arquivo}"
    s3.put_object(Bucket=BUCKET, Key=s3_key, Body=response.content)
    logger.info(f"[MICRODADOS] Upload concluído: s3://{BUCKET}/{s3_key}")


def baixar_e_subir_documentacao(nome_arquivo):
    """Baixa o dicionário XLS do FTP e sobe intacto para data_input."""
    url = f"{BASE_URL_DOC}{nome_arquivo}"
    logger.info(f"[DOCUMENTACAO] Iniciando download: {url}")

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    s3_key = f"data_input/documentacao/{nome_arquivo}"
    s3.put_object(Bucket=BUCKET, Key=s3_key, Body=response.content)
    logger.info(f"[DOCUMENTACAO] Upload concluído: s3://{BUCKET}/{s3_key}")


# -- Execucao --
erros = []

for arquivo in MICRODADOS:
    try:
        baixar_e_subir_zip(arquivo)
    except Exception as e:
        logger.error(f"[MICRODADOS] Falha em {arquivo}: {e}")
        erros.append(f"microdados/{arquivo}: {e}")

for arquivo in DOCUMENTACAO:
    try:
        baixar_e_subir_documentacao(arquivo)
    except Exception as e:
        logger.error(f"[DOCUMENTACAO] Falha em {arquivo}: {e}")
        erros.append(f"documentacao/{arquivo}: {e}")

if erros:
    raise RuntimeError(f"Job finalizado com {len(erros)} erro(s):\n" + "\n".join(erros))

logger.info("Job de coleta finalizado com sucesso.")
job.commit()

## Bronze Spark

In [ ]:
import sys
import io
import logging
import zipfile
import re

import boto3
from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext
from pyspark.sql import functions as F

# -- Logging --
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -- Glue init --
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME"])
    job_name = args["JOB_NAME"]
except:
    args = {}
    job_name = "local_test"

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init(job_name, args)

# -- Configuracaes --
s3 = boto3.client("s3")

BUCKET       = "datalake-pnad-covid-prod"
INPUT_MICRO  = "data_input/microdados/"
INPUT_DOC    = "data_input/documentacao/"
OUTPUT_MICRO = "data_output/bronze/microdados/"
OUTPUT_DOC   = "data_output/bronze/documentacao/"

MESES = ["9", "10", "11"]

MICRODADOS  = [f"PNAD_COVID_{m}2020.zip"                     for m in MESES]
DICIONARIOS = [f"Dicionario_PNAD_COVID_{m}2020_20220621.xls" for m in MESES]


# -- Helpers S3 --
def s3_get_bytes(key):
    logger.info(f"Lendo S3: s3://{BUCKET}/{key}")
    response = s3.get_object(Bucket=BUCKET, Key=key)
    return response["Body"].read()


# -- Helper mes --
def extrair_mes_ref(nome_arquivo):
    match = re.search(r"(\d{6})", nome_arquivo)
    return match.group(1) if match else None


# -- Microdados: ZIP -> DataFrame Spark -> Parquet --
def processar_microdados(nome_zip):
    """
    Refatorado:
    - Remove escrita temporária no S3
    - Usa Spark para leitura distribuída (RDD)
    - Mantem logica original da ETL
    """

    key_input = f"{INPUT_MICRO}{nome_zip}"
    logger.info(f"[BRONZE][MICRO] Processando: s3://{BUCKET}/{key_input}")

    # 1. Le ZIP (unico ponto nao distribuido - limitacao natural)
    zip_bytes = s3_get_bytes(key_input)

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        csvs = [f for f in z.namelist() if f.upper().endswith(".CSV")]
        if not csvs:
            raise ValueError(f"Nenhum CSV encontrado dentro de {nome_zip}")

        nome_csv = csvs[0]
        logger.info(f"[BRONZE][MICRO] Extraindo: {nome_csv}")

        csv_bytes = z.read(nome_csv)

    # 2. Converte para string
    csv_str = csv_bytes.decode("latin1")

    # 3. Cria RDD distribuído
    rdd = spark.sparkContext.parallelize(csv_str.split("\n"))

    # 4. Cria DataFrame Spark
    df = (
        spark.read
        .option("header", True)
        .option("sep", ",")
        .option("inferSchema", True)
        .csv(rdd)
    )

    # 5. Rastreabilidade
    mes_ref = extrair_mes_ref(nome_zip)
    df = df.withColumn("mes_ref", F.lit(mes_ref))

    # 6. Escrita otimizada
    nome_base   = nome_csv.replace(".csv", "").replace(".CSV", "")
    output_path = f"s3://{BUCKET}/{OUTPUT_MICRO}{nome_base}/"

    (
        df.write
        .mode("overwrite")
        .parquet(output_path)
    )

    logger.info(
        f"[BRONZE][MICRO] Finalizado: {output_path} | Linhas: {df.count()}"
    )


# -- Dicionarios: copia direta --
def copiar_dicionario(nome_xls):
    """
    Mantido propositalmente simples (ETL puro):
    Bronze NÃO transforma documentação.
    """

    key_input  = f"{INPUT_DOC}{nome_xls}"
    key_output = f"{OUTPUT_DOC}{nome_xls}"

    logger.info(f"[BRONZE][DOC] Copiando: s3://{BUCKET}/{key_input}")

    s3.copy_object(
        Bucket=BUCKET,
        CopySource={"Bucket": BUCKET, "Key": key_input},
        Key=key_output
    )

    logger.info(f"[BRONZE][DOC] Concluido: s3://{BUCKET}/{key_output}")


# -- Execucao --
def main():

    erros = []

    # Microdados
    for arquivo in MICRODADOS:
        try:
            processar_microdados(arquivo)
        except Exception as e:
            logger.error(f"[BRONZE][MICRO] Erro em {arquivo}: {e}")
            erros.append(f"microdados/{arquivo}: {e}")

    # Dicionarios
    for arquivo in DICIONARIOS:
        try:
            copiar_dicionario(arquivo)
        except Exception as e:
            logger.error(f"[BRONZE][DOC] Erro em {arquivo}: {e}")
            erros.append(f"documentacao/{arquivo}: {e}")

    # Controle de falha
    if erros:
        raise RuntimeError(
            f"Job finalizado com {len(erros)} erro(s):\n" + "\n".join(erros)
        )

    logger.info("Job Bronze finalizado com sucesso.")


# -- Run --
if __name__ == "__main__":
    main()
    job.commit()

## Silver Spark

In [ ]:
%additional_python_modules xlrd==2.0.1

In [ ]:
import sys
import io
import logging

import boto3
import pandas as pd

from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# -- Logging --
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -- Glue init --
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME"])
    job_name = args["JOB_NAME"]
except:
    args = {}
    job_name = "local_test"

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init(job_name, args)

# -- Configuracoes --
s3 = boto3.client("s3")

BUCKET = "datalake-pnad-covid-prod"

INPUT_MICRO  = "data_output/bronze/microdados/"
OUTPUT_MICRO = "data_output/silver/microdados/"
INPUT_DOC    = "data_output/bronze/documentacao/"

MESES = ["092020", "102020", "112020"]

# -- Mapeamento --
RENAME = {
    # Contexto temporal
    "UF":     "uf",
    "V1012":  "semana_entrevista",
    "V1013":  "mes_referencia",

    # Demografia
    "A002":   "idade",
    "A003":   "sexo",
    "A004":   "raca",
    "A005":   "escolaridade",

    # Sintomas (13)
    "B0011":  "sintoma_febre",
    "B0012":  "sintoma_tosse",
    "B0013":  "sintoma_dor_garganta",
    "B0014":  "sintoma_dificuldade_respirar",
    "B0015":  "sintoma_dor_cabeca",
    "B0016":  "sintoma_dor_peito",
    "B0017":  "sintoma_nausea",
    "B0018":  "sintoma_coriza",
    "B0019":  "sintoma_fadiga",
    "B00110": "sintoma_dor_olhos",
    "B00111": "sintoma_perda_olfato_paladar",
    "B00112": "sintoma_dor_muscular",
    "B00113": "sintoma_diarreia",

    # Atendimento
    "B002":   "buscou_atendimento",
    "B0033":  "remedio_conta_propria",

    # Locais de atendimento
    "B0041":  "atendimento_posto_ubs",
    "B0042":  "atendimento_upa_sus",
    "B0043":  "atendimento_hospital_sus",
    "B0044":  "atendimento_consultorio_privado",
    "B0045":  "atendimento_ps_privado",
    "B0046":  "atendimento_hospital_privado",

    # Internação e testes
    "B005":   "foi_internado",
    "B008":   "fez_teste_covid",
    "B009B":  "resultado_teste_swab",
    "B009D":  "resultado_teste_sangue_dedo",
    "B009F":  "resultado_teste_sangue_veia",

    # Isolamento
    "B011":   "nivel_isolamento",

    # Trabalho
    "C001":   "trabalhou_semana",
    "C004":   "recebeu_remuneracao_afastado",
    "C007D":  "setor_atividade_empresa",

    # Benefícios
    "D0051":  "recebeu_auxilio_emergencial",
    "D0061":  "recebeu_seguro_desemprego",

    # Crédito
    "E001":   "solicitou_emprestimo",

    # Itens de proteção
    "F002A2": "item_alcool_gel",
    "F002A3": "item_mascara",
}

# -- Inverso --
RENAME_INV = {novo: original for original, novo in RENAME.items()}

# -- Tipagem --
COLUNAS_INT = {"idade", "uf", "mes_referencia", "semana_entrevista"}

# -- Colunas sem lookup --
COLUNAS_SEM_LOOKUP = {"mes_referencia", "semana_entrevista", "idade"}


# -- Silver: Dicionario para lookup --
def carregar_lookup(mes):

    key = f"{INPUT_DOC}Dicionario_PNAD_COVID_{mes}_20220621.xls"
    logger.info(f"[SILVER][LOOKUP] Lendo: s3://{BUCKET}/{key}")

    xls_bytes = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()

    df = pd.read_excel(
        io.BytesIO(xls_bytes),
        engine="xlrd",
        header=None,
        skiprows=2
    )

    df.columns = [
        "tamanho", "codigo_variavel", "quesito_num",
        "quesito_descricao", "categoria_tipo", "categoria_descricao"
    ]

    df = df[df["tamanho"] != "nº"]
    df = df[~df["tamanho"].astype(str).str.startswith("Parte")]

    df["tamanho"] = df["tamanho"].ffill()
    df["codigo_variavel"] = df["codigo_variavel"].ffill()
    df["quesito_descricao"] = df["quesito_descricao"].ffill()

    df = df[df["codigo_variavel"].notna()]
    df = df.astype(str).apply(lambda x: x.str.strip())
    df = df.replace("nan", None)

    df = df[df["categoria_tipo"].notna()]

    df_lookup = spark.createDataFrame(
        df[["codigo_variavel", "categoria_tipo", "categoria_descricao"]]
    ).select(
        F.col("codigo_variavel"),
        F.trim(F.col("categoria_tipo")).alias("codigo"),
        F.col("categoria_descricao").alias("descricao")
    )

    df_lookup_pad = df_lookup.withColumn("codigo", F.lpad(F.col("codigo"), 2, "0"))

    return df_lookup.unionByName(df_lookup_pad).dropDuplicates()


# -- Aplica lookup dinamicamente --
def aplicar_lookup(df, df_lookup):

    for col_semantico, col_original in RENAME_INV.items():

        if col_semantico not in df.columns:
            continue

        if col_semantico in COLUNAS_SEM_LOOKUP:
            continue

        df = df.withColumn(
            f"{col_semantico}_key",
            F.lpad(F.trim(F.col(col_semantico)), 2, "0")
        )

        df = df.join(
            df_lookup.filter(F.col("codigo_variavel") == col_original),
            (df[f"{col_semantico}_key"] == df_lookup["codigo"]),
            "left"
        )

        df = df.withColumn(
            col_semantico,
            F.coalesce(F.col("descricao"), F.col(col_semantico))
        )

        df = df.drop(f"{col_semantico}_key", "codigo", "descricao", "codigo_variavel")

    return df


# -- Silver: Microdados --
def processar_microdados(mes):

    input_path = f"s3://{BUCKET}/{INPUT_MICRO}PNAD_COVID_{mes}/"
    output_path = f"s3://{BUCKET}/{OUTPUT_MICRO}PNAD_COVID_{mes}/"

    logger.info(f"[SILVER][MICRO] Lendo: {input_path}")

    df = spark.read.parquet(input_path)

    colunas_existentes = [c for c in RENAME.keys() if c in df.columns]
    colunas_ausentes = [c for c in RENAME.keys() if c not in df.columns]

    if colunas_ausentes:
        logger.warning(f"[SILVER][MICRO] Colunas ausentes: {colunas_ausentes}")

    df = df.select(colunas_existentes)

    for original, novo in RENAME.items():
        if original in df.columns:
            df = df.withColumnRenamed(original, novo)

    df = df.select([
        F.when(F.col(c).isin("", "99"), None).otherwise(F.col(c)).alias(c)
        for c in df.columns
    ])

    for col_name in COLUNAS_INT:
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(IntegerType()))

    df_lookup = carregar_lookup(mes)
    df = aplicar_lookup(df, df_lookup)

    df.coalesce(1).write.mode("overwrite").parquet(output_path)

    logger.info(
        f"[SILVER][MICRO] Finalizado: {output_path} | Linhas: {df.count()}"
    )


# -- Execucao --
def main():

    erros = []

    for mes in MESES:
        try:
            processar_microdados(mes)
        except Exception as e:
            logger.error(f"[SILVER] Erro em {mes}: {e}")
            erros.append(f"silver/{mes}: {e}")

    if erros:
        raise RuntimeError(
            f"Job finalizado com {len(erros)} erro(s):\n" + "\n".join(erros)
        )

    logger.info("Job Silver finalizado com sucesso.")


# -- Run --
if __name__ == "__main__":
    main()
    job.commit()

## Gold Spark

In [ ]:
import sys
import logging

from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext

# -- Logging --
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -- Glue init --
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME"])
    job_name = args["JOB_NAME"]
except:
    args = {}
    job_name = "local_test"

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init(job_name, args)

# -- Configuracoes --
BUCKET = "datalake-pnad-covid-prod"
INPUT_MICRO = "data_output/silver/microdados/"
OUTPUT_GOLD = "data_output/gold/"
MESES = ["092020", "102020", "112020"]


# -- Processamento principal (somente consolidacao) --
def processar_gold():

    dfs = []

    for mes in MESES:
        logger.info(f"[GOLD] Processando mes: {mes}")

        micro_path = f"s3://{BUCKET}/{INPUT_MICRO}PNAD_COVID_{mes}/"
        df = spark.read.parquet(micro_path)

        dfs.append(df)

        logger.info(f"[GOLD] Mes {mes}: {df.count()} linhas")

    df_gold = dfs[0]
    for df_mes in dfs[1:]:
        df_gold = df_gold.unionByName(df_mes)

    output_path = f"s3://{BUCKET}/{OUTPUT_GOLD}pnad_covid_consolidado/"

    df_gold = df_gold.coalesce(1)

    df_gold.write \
        .mode("overwrite") \
        .parquet(output_path)

    logger.info(
        f"[GOLD] Finalizado: {output_path} | Linhas: {df_gold.count()} | Colunas: {len(df_gold.columns)}"
    )


# -- Execucao --
try:
    processar_gold()
except Exception as e:
    logger.error(f"[GOLD] Falha: {e}")
    raise

logger.info("Job Gold finalizado com sucesso.")
job.commit()

ModuleNotFoundError: No module named 'awsglue'